# Quanto o pré-processamento ajuda?

Estudo pareado: full (imagem inteira), whole (ROI com fundo), gray (máscara rígida), blur (fundo desfocado), context (máscara dilatada).
Todas as variantes usam as mesmas origens, grupos e resolução. A caixa vem da mesma máscara congelada. Não há alinhamento nem limpeza de máscara nesta primeira ablação.

Escolha as condições antes de avaliar o teste. O split `val` é reservado à calibração normal, não à seleção de modelos. Dados usados para desenvolver estas escolhas devem ser separados do teste final.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from insulator_anomaly.config import ProjectConfig
from insulator_anomaly.crop_experiment import CropSettings, run_crop_experiment
from insulator_anomaly.preprocessing_study import MODES, export_masks, prepare_study, paired_effects

config = ProjectConfig(Path("anomalib_glass"), accelerator="gpu")
config.segmenter = "yolo"  # fixar pesos/segmentador; não treinar com o teste
CACHE = Path("experiments/preprocessing_masks_v1")
RUN = Path("experiments/preprocessing_v1")
GROUPS_PATH = Path("groups.json")
settings = CropSettings(output_size=256, margin=0.20, context_ratio=0.05,
                        validation_fraction=0.20, normal_fpr=0.05, seed=42)
RUN_MASKS = False
RUN_PREPARATION = False
RUN_TRAINING = False


## 1. Grupos e máscaras
Crie `groups.json`: cada chave é um caminho relativo ao dataset bruto (ex.: `train/good/foto.jpg`) e cada valor identifica a peça/cadeia/sessão. Todos os arquivos devem ter um grupo. Nunca coloque imagens relacionadas em grupos diferentes para contornar uma sobreposição.

`export_masks` salva máscaras binárias na resolução original e os hashes de origem/máscara. Não usa anotações de defeito. Falta de detecção gera máscara vazia; todas as condições usam então a imagem inteira, sem descartar a amostra. Outros erros interrompem a execução.


In [ ]:
if RUN_MASKS:
    masks = export_masks(config, CACHE)
    display(masks.groupby("status").size())


In [ ]:
if RUN_PREPARATION:
    groups = json.loads(GROUPS_PATH.read_text(encoding="utf-8"))
    manifest = prepare_study(config, RUN, CACHE / "masks.csv", groups, settings)
    display(manifest.groupby(["mode", "split", "y_true"]).size())
    display(manifest.groupby("mode").segmentation_failed.mean())


## 2. Conferir as entradas antes de treinar
Escolha exemplos do treino para conferir a máscara e a geometria. Todas as variantes com recorte devem ter a mesma caixa e escala. `full` versus `whole` mede localização + mudança de escala; `whole` versus fundos isola o tratamento do fundo.


In [ ]:
if (RUN / "manifest.csv").exists():
    import matplotlib.pyplot as plt
    from insulator_anomaly.preprocessing import read_image
    manifest = pd.read_csv(RUN / "manifest.csv")
    source = manifest.loc[manifest.split == "train", "source_id"].iloc[0]
    fig, axes = plt.subplots(1, len(MODES), figsize=(16, 4))
    for ax, mode in zip(axes, MODES):
        row = manifest[(manifest.source_id == source) & (manifest["mode"] == mode)].iloc[0]
        ax.imshow(read_image(Path(row.image_path))[..., ::-1])
        ax.set_title(mode)
        ax.axis("off")
    plt.show()


## 3. Treinar um PatchCore independente por variante
Mesmo backbone, resolução, orçamento de memória e seed. Checkpoints compatíveis são reutilizados. O limiar de cada modelo é fixado no holdout normal; reporte a FPR observada no teste, não apenas a meta.


In [ ]:
if RUN_TRAINING:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("Instale PyTorch com CUDA no kernel antes do treino.")
    comparison, decisions = run_crop_experiment(RUN, accelerator="gpu", modes=MODES)
    display(comparison)


## 4. Efeito pareado e incerteza
Deltas = variante menos baseline. Positivo é melhor para recall/AP/AUROC; negativo é melhor para FPR. IC95 por bootstrap de grupos, condicionado aos modelos e limiares ajustados. Não corrige comparações múltiplas nem substitui repetições de treino. Não escolher o vencedor com base no teste final.


In [ ]:
if (RUN / "source_scores.csv").exists():
    scores = pd.read_csv(RUN / "source_scores.csv", dtype={"source_id": str, "group_id": str})
    protocol = json.loads((RUN / "protocol.json").read_text(encoding="utf-8"))
    effects = paired_effects(scores, baseline="whole",
                             normal_fpr=protocol["settings"]["normal_fpr"])
    effects.to_csv(RUN / "paired_effects.csv", index=False)
    display(effects)


## Limites e próximos experimentos
- As medições de preparação em `manifest.csv` são diagnósticas, sem aquecimento, e não incluem inferência/segmentação ponta a ponta; não são benchmark de latência para publicação.
- Não há máscaras de defeito neste estudo inicial: não calcula preservação de defeito, IoU nem AUPRO.
- Repita em outro destino para resolução/margem; mantenha seed de split fixa. Alterar a seed atual também muda o holdout, portanto não representa só variabilidade de treinamento.
- Se os dados históricos já foram inspecionados para escolher o método, os resultados são exploratórios. Reserve peças novas para confirmação.
